# Spatial Validation

Random cross-validation and spatially separated validation answer different questions. This notebook compares random folds, contiguous held-out stripes, and stripes with a buffer that removes nearby training observations.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import KFold

rng = np.random.default_rng(5)
n = 600
coords = rng.uniform(0,1,size=(n,2))
x, y = coords[:,0], coords[:,1]
target = (
    np.sin(3*np.pi*x)
    + np.cos(3*np.pi*y)
    + 0.6*np.sin(4*np.pi*(x+y))
    + rng.normal(scale=0.18, size=n)
)


In [ ]:
def model():
    return RandomForestRegressor(
        n_estimators=180, min_samples_leaf=3, random_state=11, n_jobs=1
    )

def stripe_splits(buffer=0.0, n_splits=5):
    edges = np.linspace(0,1,n_splits+1)
    for f in range(n_splits):
        left, right = edges[f], edges[f+1]
        test = (x >= left) & ((x < right) if f < n_splits-1 else (x <= right))
        train = ~test
        if buffer:
            train &= (x < left-buffer) | (x > right+buffer)
        yield np.flatnonzero(train), np.flatnonzero(test)

def evaluate(splits):
    scores = []
    for train, test in splits:
        m = model()
        m.fit(coords[train], target[train])
        pred = m.predict(coords[test])
        scores.append(np.sqrt(mean_squared_error(target[test], pred)))
    return np.array(scores)

random = evaluate(KFold(5, shuffle=True, random_state=13).split(coords))
blocked = evaluate(stripe_splits(0.0))
buffered = evaluate(stripe_splits(0.04))

results = pd.DataFrame({
    "scheme":["random folds","spatial blocks","buffered blocks"],
    "mean RMSE":[random.mean(),blocked.mean(),buffered.mean()],
    "sd RMSE":[random.std(ddof=1),blocked.std(ddof=1),buffered.std(ddof=1)]
})
results


In [ ]:
fig, ax = plt.subplots(figsize=(6,4))
ax.bar(results["scheme"], results["mean RMSE"], yerr=results["sd RMSE"])
ax.set(ylabel="RMSE", title="Validation difficulty changes with spatial separation")
ax.tick_params(axis="x", rotation=20)
plt.show()


The random folds approximate interpolation because test points are usually surrounded by training points. Contiguous and buffered holdouts ask progressively harder transfer questions.

The best split is not the one that gives the lowest error. It is the one that reproduces the spatial separation and information availability expected at deployment.